In [19]:
import mlflow
from mlflow.models import infer_signature

import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [20]:
mlflow.__version__

'2.22.0'

In [21]:
# Load the Iris dataset
X, y = datasets.load_iris(return_X_y=True)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [22]:
# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 200,
    "multi_class": "multinomial",
    "random_state": 56,
}

In [23]:
clf = DecisionTreeClassifier(max_depth=2, criterion='gini')
clf.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=2)

In [24]:
# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

/opt/miniconda3/envs/otus_new/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=200, multi_class='multinomial', random_state=56)

In [25]:
# Predict on the test set
y_pred_lr = lr.predict(X_test)
y_pred_clf = clf.predict(X_test)

In [26]:
# Calculate metrics
accuracy_lr = accuracy_score(y_test, y_pred_lr)
accuracy_clf = accuracy_score(y_test, y_pred_clf)
accuracy_lr, accuracy_clf

(1.0, 0.9666666666666667)

Запустим сервер MLFlow в локальной папке

```bash
mlflow server \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns \
  --host 127.0.0.1 \
  --port 8080
```

In [27]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

# Create a new MLflow Experiment
mlflow.set_experiment("MLflow Open lesson MLOps")

<Experiment: artifact_location='/Users/stureiko/Documents/Programming/Otus/MLOps/Open_day_MLFlow/code/mlruns/1', creation_time=1765207687103, experiment_id='1', last_update_time=1765207687103, lifecycle_stage='active', name='MLflow Open lesson MLOps', tags={}>

In [10]:
params

{'solver': 'lbfgs',
 'max_iter': 200,
 'multi_class': 'multinomial',
 'random_state': 56}

In [29]:
# Start an MLflow run
with mlflow.start_run(run_name='Second run'):
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("LogisticRegression accuracy", accuracy_lr)   #type: ignore
    mlflow.log_metric("DecisionTreeClassifier accuracy", accuracy_clf)  #type: ignore

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model for iris data")

    # Infer the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(  #type: ignore
        sk_model=lr,
        artifact_path="iris_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="tracking-quickstart",
    )


Registered model 'tracking-quickstart' already exists. Creating a new version of this model...
2025/12/08 20:47:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 8


🏃 View run Second run at: http://127.0.0.1:8080/#/experiments/1/runs/f394865fb79c4c3bb23517f64f733169
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


Created version '8' of model 'tracking-quickstart'.


## Создадим еще один эксперимент с двумя запусками

In [30]:
# Create a new MLflow Experiment
mlflow.set_experiment("MLflow Experiment 2")
param_list = [{"solver": "lbfgs",\
    "max_iter": 200,\
    "multi_class": "multinomial",\
    "random_state": 56,\
},\
{"solver": "lbfgs",\
    "max_iter": 30,\
    "multi_class": "multinomial",\
    "random_state": 23,\
}]
# Start an MLflow run
for i in range(0, 2):
    with mlflow.start_run(run_name=f'Run {i+1}'):
        # Log the hyperparameters
        mlflow.log_params(param_list[i])
        # Log the loss metric
        mlflow.log_metric("LogisticRegression accuracy", accuracy_lr)
        mlflow.log_metric("DecisionTreeClassifier accuracy", accuracy_clf)

        # Set a tag that we can use to remind ourselves what this run was for
        mlflow.set_tag("Training Info", "Basic LR model for iris data")

        # Infer the model signature
        signature = infer_signature(X_train, lr.predict(X_train))

        # Log the model
        model_info = mlflow.sklearn.log_model(
            sk_model=lr,
            artifact_path="iris_model",
            signature=signature,
            input_example=X_train,
            registered_model_name="tracking-quickstart",
        )

Registered model 'tracking-quickstart' already exists. Creating a new version of this model...
2025/12/08 20:50:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 9
Created version '9' of model 'tracking-quickstart'.


🏃 View run Run 1 at: http://127.0.0.1:8080/#/experiments/2/runs/dee6a1b09b444e69a6c93aa4947f34b9
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/2


Registered model 'tracking-quickstart' already exists. Creating a new version of this model...
2025/12/08 20:50:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 10


🏃 View run Run 2 at: http://127.0.0.1:8080/#/experiments/2/runs/b7d43e6d58464c12b6f64b7d30353dc9
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/2


Created version '10' of model 'tracking-quickstart'.


## Получим список экспериментов

In [31]:
from mlflow.tracking import MlflowClient

In [32]:
client = MlflowClient()
experiments = client.search_experiments()  # по умолчанию только active_only=True

for exp in experiments:
    print(f"ID={exp.experiment_id}, name={exp.name}, artifact_location={exp.artifact_location}")

ID=2, name=MLflow Experiment 2, artifact_location=/Users/stureiko/Documents/Programming/Otus/MLOps/Open_day_MLFlow/code/mlruns/2
ID=1, name=MLflow Open lesson MLOps, artifact_location=/Users/stureiko/Documents/Programming/Otus/MLOps/Open_day_MLFlow/code/mlruns/1
ID=0, name=Default, artifact_location=/Users/stureiko/Documents/Programming/Otus/MLOps/Open_day_MLFlow/code/mlruns/0


## Получим все запуски внутри эксперимента ID=2, name=MLflow Experiment 2

In [33]:
experiment_id = 2

runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="",           # можно задать фильтр по метрикам/парам
    order_by=["metrics.auc DESC"],  # сортировка по метрике
    max_results=1000,
)

for run in runs:
    info = run.info
    data = run.data

    print("=== RUN ===")
    print("run_id:", info.run_id)
    print("status:", info.status)
    print("start_time:", info.start_time)
    print("end_time:", info.end_time)

    # Метрики
    print("metrics:")
    for k, v in data.metrics.items():
        print(f"  {k}: {v}")

    # Гиперпараметры
    print("params:")
    for k, v in data.params.items():
        print(f"  {k}: {v}")

    # Теги при необходимости
    print("tags:")
    for k, v in data.tags.items():
        print(f"  {k}: {v}")

    print()

=== RUN ===
run_id: b7d43e6d58464c12b6f64b7d30353dc9
status: FINISHED
start_time: 1765216202812
end_time: 1765216208462
metrics:
  LogisticRegression accuracy: 1.0
  DecisionTreeClassifier accuracy: 0.9666666666666667
params:
  solver: lbfgs
  max_iter: 30
  multi_class: multinomial
  random_state: 23
tags:
  mlflow.user: stureiko
  mlflow.source.name: /opt/miniconda3/envs/otus_new/lib/python3.12/site-packages/ipykernel_launcher.py
  mlflow.source.type: LOCAL
  mlflow.runName: Run 2
  Training Info: Basic LR model for iris data
  mlflow.log-model.history: [{"run_id": "b7d43e6d58464c12b6f64b7d30353dc9", "artifact_path": "iris_model", "utc_time_created": "2025-12-08 17:50:02.863756", "model_uuid": "8539fae9482740d7a2d48fbf0304c94c", "flavors": {"python_function": {"model_path": "model.pkl", "predict_fn": "predict", "loader_module": "mlflow.sklearn", "python_version": "3.12.3", "env": {"conda": "conda.yaml", "virtualenv": "python_env.yaml"}}, "sklearn": {"pickled_model": "model.pkl", "skl

## Ссылки на сохраненные модели

In [34]:
for run in runs:
    info = run.info
    print("run_id:", info.run_id)
    print("artifact_uri:", info.artifact_uri)

run_id: b7d43e6d58464c12b6f64b7d30353dc9
artifact_uri: /Users/stureiko/Documents/Programming/Otus/MLOps/Open_day_MLFlow/code/mlruns/2/b7d43e6d58464c12b6f64b7d30353dc9/artifacts
run_id: dee6a1b09b444e69a6c93aa4947f34b9
artifact_uri: /Users/stureiko/Documents/Programming/Otus/MLOps/Open_day_MLFlow/code/mlruns/2/dee6a1b09b444e69a6c93aa4947f34b9/artifacts
run_id: 8f7fabf268f84e6a93dc4f549b0d0bc9
artifact_uri: /Users/stureiko/Documents/Programming/Otus/MLOps/Open_day_MLFlow/code/mlruns/2/8f7fabf268f84e6a93dc4f549b0d0bc9/artifacts
run_id: f9359ea6091e498d8e9d4517f99cdc75
artifact_uri: /Users/stureiko/Documents/Programming/Otus/MLOps/Open_day_MLFlow/code/mlruns/2/f9359ea6091e498d8e9d4517f99cdc75/artifacts
run_id: ca8020474ca6458e871597d09cce65a3
artifact_uri: /Users/stureiko/Documents/Programming/Otus/MLOps/Open_day_MLFlow/code/mlruns/2/ca8020474ca6458e871597d09cce65a3/artifacts
run_id: b56c6cbedac1478aaa7f2a69042a69ac
artifact_uri: /Users/stureiko/Documents/Programming/Otus/MLOps/Open_day_ML

In [17]:
# import mlflow.pyfunc

In [35]:
run_id = runs[0].info.run_id  # например берем лучший

model_uri = f"runs:/{run_id}/iris_model"
print("model_uri:", model_uri)

loaded_model = mlflow.pyfunc.load_model(model_uri)

model_uri: runs:/b7d43e6d58464c12b6f64b7d30353dc9/iris_model
